In [4]:
import pandas as pd

df = pd.read_csv("../Data_Frame/data/metadata_negatives.csv")

df.sample(10)

,fname,labels,mids,split,audio_path
2376,331327,"Male_speech_and_man_speaking,Fixed-wing_aircra...","/m/05zppz,/m/0cmf2,/m/02mk9,/m/09x0r,/m/09l8g,...",NaN,data/FSD50K/FSD50K.eval_audio_16k/
1365,431299,"Rain,Water","/m/06mb1,/m/0838f",train,data/FSD50K/FSD50K.dev_audio_16k/
2363,132902,"Child_speech_and_kid_speaking,Conversation,Spe...","/m/0ytgt,/m/01h8n0,/m/09x0r,/m/09l8g",NaN,data/FSD50K/FSD50K.eval_audio_16k/
2144,371153,"Female_speech_and_woman_speaking,Gull_and_seag...","/m/02zsn,/m/01dwxx,/m/034srq,/m/09x0r,/m/09l8g...",NaN,data/FSD50K/FSD50K.eval_audio_16k/
1476,33564,"Applause,Chatter,Cheering,Clapping,Shout,Human...","/m/028ght,/m/07rkbfh,/m/053hz1,/m/0l15bq,/m/07...",NaN,data/FSD50K/FSD50K.eval_audio_16k/
1975,106789,"Female_speech_and_woman_speaking,Speech,Human_...","/m/02zsn,/m/09x0r,/m/09l8g",train,data/FSD50K/FSD50K.dev_audio_16k/
2128,201914,"Male_speech_and_man_speaking,Singing,Speech,Hu...","/m/05zppz,/m/015lz1,/m/09x0r,/m/09l8g",NaN,data/FSD50K/FSD50K.eval_audio_16k/
91,154470,"Cutlery_and_silverware,Domestic_sounds_and_hom...","/m/023pjk,/t/dd00071",NaN,data/FSD50K/FSD50K.eval_audio_16k/
1729,196899,"Speech,Human_voice","/m/09x0r,/m/09l8g",train,data/FSD50K/FSD50K.dev_audio_16k/
571,194663,"Keys_jangling,Domestic_sounds_and_home_sounds","/m/03v3yw,/t/dd00071",train,data/FSD50K/FSD50K.dev_audio_16k/


In [5]:
# 1. Definiere die Gruppen
similar_sounds = [
    "Chink_and_clink", "Coin_(dropping)", "Cutlery_and_silverware", 
    "Dishes_and_pots_and_pans", "Keys_jangling", "Crack", 
    "Crackle", "Crumpling_and_crinkling", "Crushing", "Tearing"
]

ambient_sounds = [
    "Mechanical_fan", "Traffic_noise_and_roadway_noise", "Wind", 
    "Rain", "Chatter", "Crowd"
]

def analyze_negatives(target_list, group_name):
    print(f"\n--- Analyse: {group_name} ---")
    print(f"{'Label':<35} | {'Anzahl (ohne Glas)':<15}")
    print("-" * 55)
    
    for label in target_list:
        # Filter: Label vorhanden UND "Glass" oder "Shatter" NICHT vorhanden
        mask = df['labels'].apply(lambda x: label in x.split(',') and 
                                 not any(g in x.split(',') for g in ['Glass', 'Shatter']))
        count = df[mask].shape[0]
        print(f"{label:<35} | {count:<15}")

# Ausführung
analyze_negatives(similar_sounds, "ÄHNLICHE GERÄUSCHE (Hard Negatives)")
analyze_negatives(ambient_sounds, "UMGEBUNGSGERÄUSCHE (Background)")


--- Analyse: ÄHNLICHE GERÄUSCHE (Hard Negatives) ---
Label                               | Anzahl (ohne Glas)
-------------------------------------------------------
Chink_and_clink                     | 0              
Coin_(dropping)                     | 142            
Cutlery_and_silverware              | 104            
Dishes_and_pots_and_pans            | 108            
Keys_jangling                       | 82             
Crack                               | 55             
Crackle                             | 102            
Crumpling_and_crinkling             | 76             
Crushing                            | 70             
Tearing                             | 117            

--- Analyse: UMGEBUNGSGERÄUSCHE (Background) ---
Label                               | Anzahl (ohne Glas)
-------------------------------------------------------
Mechanical_fan                      | 45             
Traffic_noise_and_roadway_noise     | 106            
Wind                  

In [7]:
from pathlib import Path
import shutil
import pandas as pd

df = pd.read_csv("../Data_Frame/data/metadata_negatives.csv")

similar_sounds = [
    "Chink_and_clink", "Coin_(dropping)", "Cutlery_and_silverware",
    "Dishes_and_pots_and_pans", "Keys_jangling", "Crack",
    "Crackle", "Crumpling_and_crinkling", "Crushing", "Tearing"
]

target_dir = Path("../Data_Frame/data/FSD50K/negative_samples_neu")
target_dir.mkdir(parents=True, exist_ok=True)

copied_files = set()
copied_count = 0

for _, row in df.iterrows():
    labels = row["labels"].split(",")

    is_hard_negative = (
        any(label in labels for label in similar_sounds)
        and not any(g in labels for g in ["Glass", "Shatter"])
    )

    if not is_hard_negative:
        continue

    source_dir = Path("../Data_Frame/data/FSD50K/negative_samples")
    src = source_dir / f'{row["fname"]}.wav'

    if src.exists() and src.name not in copied_files:
        shutil.copy2(src, target_dir / src.name)
        copied_files.add(src.name)
        copied_count += 1

print(f"{copied_count} Hard-Negative-Dateien nach {target_dir} kopiert.")

816 Hard-Negative-Dateien nach ../Data_Frame/data/FSD50K/negative_samples_neu kopiert.
